# Exploratory Data Analysis

## Part A: JSON data

### Read JSON
This section reads the GeoJSON-style files in the `data/` folder (`locations (2).json` and `roadSegments (2).json`) and loads them into pandas and GeoPandas for analysis.

In [1]:
from pathlib import Path
import json
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString

# Paths (notebook is in `notebooks/`)
DATA_DIR = Path('..') / 'data'
LOC_FP = DATA_DIR / 'locations (2).json'
ROAD_FP = DATA_DIR / 'roadSegments (2).json'

assert LOC_FP.exists(), f"Missing: {LOC_FP}"
assert ROAD_FP.exists(), f"Missing: {ROAD_FP}"

def load_geojson_features(fp):
    with open(fp, 'r', encoding='utf-8') as fh:
        obj = json.load(fh)
    return obj.get('features', [])

loc_features = load_geojson_features(LOC_FP)
road_features = load_geojson_features(ROAD_FP)

print('locations features:', len(loc_features))
print('roadSegments features:', len(road_features))

locations features: 20829
roadSegments features: 1960


In [2]:
# Create pandas DataFrames from the features; keep nested properties flattened for analysis.
# For locations we extract properties + coordinates; for road segments we extract properties + coordinate lists.
def features_to_df(features):
    # properties may be missing keys across features; json_normalize handles this reasonably
    rows = []
    for f in features:
        props = f.get('properties', {}).copy()
        # bring some top-level fields into the row (id, road_id, region_id)
        props['_id'] = f.get('_id')
        props['road_id'] = f.get('road_id')
        props['region_id'] = f.get('region_id')
        geom = f.get('geometry') or {}
        props['geometry_type'] = geom.get('type')
        props['geometry_coords'] = geom.get('coordinates')
        rows.append(props)
    return pd.DataFrame(rows)

df_loc = features_to_df(loc_features)
df_road = features_to_df(road_features)

# For locations, split coords into lon/lat if available
if 'geometry_coords' in df_loc.columns:
    df_loc['lon'] = df_loc['geometry_coords'].apply(lambda c: c[0] if isinstance(c, (list, tuple)) and len(c) >= 2 else None)
    df_loc['lat'] = df_loc['geometry_coords'].apply(lambda c: c[1] if isinstance(c, (list, tuple)) and len(c) >= 2 else None)

# For roads we leave the coordinate lists as-is, but we can get segment length from properties if present
display_cols = ['_id','road_id','region_id','geometry_type']
print('locations DataFrame shape:', df_loc.shape)
print('roadSegments DataFrame shape:', df_road.shape)
df_loc.head(3)[display_cols + ['lon','lat']]

locations DataFrame shape: (20829, 13)
roadSegments DataFrame shape: (1960, 45)


,_id,road_id,region_id,geometry_type,lon,lat
0,68d703342294ad50adb9ebbf,68d703332294ad50adb9eaa8,usa_georgia_peachtree-corners,Point,-84.225030,33.961139
1,68d703342294ad50adb9ebc0,68d703332294ad50adb9eaa8,usa_georgia_peachtree-corners,Point,-84.225179,33.961143
2,68d703342294ad50adb9ebc1,68d703332294ad50adb9eaa8,usa_georgia_peachtree-corners,Point,-84.225328,33.961179


In [3]:
# Convert to GeoDataFrames
# locations -> points
loc_gdf = gpd.GeoDataFrame(df_loc.copy(), geometry=[Point(xy) if xy is not None else None for xy in df_loc['geometry_coords'].apply(lambda c: (c[0], c[1]) if isinstance(c, (list,tuple)) and len(c)>=2 else None)])
loc_gdf.set_crs('EPSG:4326', inplace=True)
print('Converted locations to GeoDataFrame, crs=', loc_gdf.crs)

# road segments -> LineString geometries when possible
def coords_to_linestring(coords):
    try:
        return LineString(coords)
    except Exception:
        return None

road_geom = df_road['geometry_coords'].apply(lambda c: coords_to_linestring(c) if isinstance(c, list) else None)
road_gdf = gpd.GeoDataFrame(df_road.copy(), geometry=road_geom)
road_gdf.set_crs('EPSG:4326', inplace=True)
print('Converted roadSegments to GeoDataFrame, sample:')
display(road_gdf.head(2))

Converted locations to GeoDataFrame, crs= EPSG:4326
Converted roadSegments to GeoDataFrame, sample:


,name,centerline_id,speed_limit,FROMLEFT,TOLEFT,FROMRIGHT,TORIGHT,FEDROUTE,FEDRTETYPE,STROUTE,...,osmid,_id,road_id,region_id,geometry_type,geometry_coords,pci,pci_category,raw_pci,geometry
0,RIVER TRAIL DR,223312,25,4430,4458,4431,4459,None,None,None,...,1000,68d703322294ad50adb9e3ff,None,usa_georgia_peachtree-corners,LineString,"[[-84.22776889587401, 34.00224609386214], [-84...",NaN,NaN,NaN,"LINESTRING (-84.22777 34.00225, -84.22777 34.0..."
1,PEACHTREE INDUSTRIAL BLVD ACCESS RD,222121,55,6450,6458,0,0,None,None,None,...,1000,68d703322294ad50adb9e400,None,usa_georgia_peachtree-corners,LineString,"[[-84.2414262608243, 33.943113373539326], [-84...",82.0,Very Good,179.104604,"LINESTRING (-84.24143 33.94311, -84.24056 33.9..."
